# Customer Churn Prediction
This notebook demonstrates how to predict customer churn using:
1. **XGBoost**: A powerful gradient boosting algorithm.
2. **SMOTE**: To handle class imbalance (more non-churners than churners).
3. **Threshold Analysis**: To optimize the classification threshold for business needs.
4. **SHAP values**: To explain feature importance and how features impact the model.


In [4]:
!pip install xgboost imbalanced-learn shap scikit-learn pandas numpy matplotlib seaborn



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

# 1. Create a synthetic dataset for customer churn
np.random.seed(42)
n_samples = 1500

# Features: Tenure (months), MonthlyCharges, Age, SupportTickets
tenure = np.random.randint(1, 72, n_samples)
monthly_charges = np.random.uniform(20, 120, n_samples)
age = np.random.randint(18, 80, n_samples)
support_tickets = np.random.randint(0, 10, n_samples)

# Churn logic: High churn if low tenure, high charges, and high support tickets
churn_prob = (72 - tenure) / 72 * 0.3 + (monthly_charges) / 120 * 0.3 + (support_tickets) / 10 * 0.3 + (age > 60) * 0.1

# Let's artificially make it imbalanced (e.g., ~15% churn)
churn = np.where(churn_prob > 0.7, 1, 0) 

data = pd.DataFrame({'Tenure': tenure, 'MonthlyCharges': monthly_charges, 'Age': age, 'SupportTickets': support_tickets, 'Churn': churn})

print("--- Class Distribution Before SMOTE ---")
print(data['Churn'].value_counts(normalize=True) * 100)


--- Class Distribution Before SMOTE ---
Churn
0    90.6
1     9.4
Name: proportion, dtype: float64


d:\Vignesh repositories\data_science_learn\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 2. Splitting the data
X = data.drop('Churn', axis=1)
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Applying SMOTE to handle imbalance
# SMOTE will generate synthetic data points for the minority class (Churn=1) so the model learns it better
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\n--- Class Distribution After SMOTE ---")
print(pd.Series(y_train_smote).value_counts(normalize=True) * 100)

In [ ]:
# 4. Train XGBoost Classifier
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_smote, y_train_smote)

# Make predictions
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print("\n--- Classification Report (Default Threshold 0.5) ---")
print(classification_report(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred_proba))

In [ ]:
# 5. Threshold Analysis
# In churn, missing a churner is expensive. We might want to lower the threshold 
# below 0.5 to catch more potential churners (increasing Recall), even if it means 
# some false positives (sending retention offers to people who wouldn't leave).

custom_threshold = 0.35
y_pred_custom = (y_pred_proba >= custom_threshold).astype(int)

print(f"\n--- Classification Report (Custom Threshold {custom_threshold}) ---")
print(classification_report(y_test, y_pred_custom))

from sklearn.metrics import ConfusionMatrixDisplay
cm = confusion_matrix(y_test, y_pred_custom)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix at Threshold {custom_threshold}')
plt.show()

In [ ]:
# 6. SHAP Feature Importance
# SHAP helps us explain HOW the model made its predictions.
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test)

print("\n--- SHAP Summary Plot (Feature Importance) ---")
# The beeswarm plot shows both importance (Y axis) and impact direction (color)
shap.summary_plot(shap_values, X_test)